In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split,cross_validate
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


In [4]:
# Loading the dataset

data = pd.read_csv(r'C:\Users\Vineet\OneDrive\Desktop\Customer_Churn_Prediction_Resume\Data\customerchurn.csv')


In [5]:
# Data Cleaning

data['TotalCharges'] = data['TotalCharges'].replace(' ', '0')
data['TotalCharges'] = data['TotalCharges'].astype(float)


In [6]:
#  Convert target variable
#    No  -> 0
#    Yes -> 1

data['Churn'] = data['Churn'].map({
    'No': 0,
    'Yes': 1
})


In [7]:
# Remove customerID

data = data.drop(columns=['customerID'])

In [8]:
# 5. Define numerical and categorical columns


numerical_columns = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

categorical_columns = [
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod'
]


In [ ]:
# Train-test split


data_train, data_test = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data['Churn']
)

In [10]:
# Separate X and y

X_train = data_train.drop(columns=['Churn'])
y_train = data_train['Churn']

X_test = data_test.drop(columns=['Churn'])
y_test = data_test['Churn']


In [11]:
#  Preprocessing

preprocessing = make_column_transformer(
    (StandardScaler(), numerical_columns),
    (OneHotEncoder(handle_unknown='ignore'), categorical_columns)
)



In [12]:
# Define models

models = [
    LogisticRegression(max_iter=1000),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    XGBClassifier()
]

In [13]:
# Cross-validation

results = []

for model in models:

    final_pipeline = make_pipeline(
        preprocessing,
        model
    )

    scores = cross_validate(
        final_pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=['recall', 'f1']
    )

    results.append([
        model.__class__.__name__,
        scores['test_recall'].mean(),
        scores['test_f1'].mean()
    ])

for result in results:
    print(result)


['LogisticRegression', np.float64(0.5484949832775919), np.float64(0.5982084032697867)]
['DecisionTreeClassifier', np.float64(0.48762541806020065), np.float64(0.4778039051556253)]
['RandomForestClassifier', np.float64(0.4782608695652174), np.float64(0.5416987244034012)]
['XGBClassifier', np.float64(0.517056856187291), np.float64(0.5560295261898001)]


In [14]:

# Display Results (Logistic Regression performs the best out of all the models)

for model_name, mean_recall , mean_f1 in results:

    print(model_name)
    print("Mean Recall:",mean_recall)
    print("Mean F1:", mean_f1)
    print("-" * 50)

LogisticRegression
Mean Recall: 0.5484949832775919
Mean F1: 0.5982084032697867
--------------------------------------------------
DecisionTreeClassifier
Mean Recall: 0.48762541806020065
Mean F1: 0.4778039051556253
--------------------------------------------------
RandomForestClassifier
Mean Recall: 0.4782608695652174
Mean F1: 0.5416987244034012
--------------------------------------------------
XGBClassifier
Mean Recall: 0.517056856187291
Mean F1: 0.5560295261898001
--------------------------------------------------
